# Biohub 3D Cell Tracking — Model Predictions & Tracking Visualization

Loads the trained 3D U-Net model (`ResidualCellTrackingUNet`), runs inference with valid-padding spatial offset corrections and spatial reflection padding, and **renders predicted cell centroids and tracking trajectory edges directly on top of microscopy images**.

In [ ]:
# ─── Imports & Environment Setup ─────────────────────────────────────────────────────────────
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import blosc2
import json
import glob

# Ensure project root is in sys.path
if '.' not in sys.path:
    sys.path.insert(0, '.')


In [ ]:
# ─── Load Model & Configuration ─────────────────────────────────────────────────────────────
from src.config import Config, ModelConfig, DataConfig, InferenceConfig
from src.models.residual_unet3d import ResidualCellTrackingUNet
from src.infer import compute_crop_offset, pad_spatial_tensor_if_needed
from src.tracking.linker import extract_peaks_from_frame, build_tracking_graph

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

config = Config(
    model=ModelConfig(
        in_channels=1,
        base_features=12,
        feature_multiplier=3.0,
        downsample_factors=[[1, 2, 2], [2, 2, 2], [2, 2, 2]],
        nms_kernel=5,
        nms_threshold=0.4,
    ),
    data=DataConfig(
        input_shape=[64, 256, 256],
        scale=[1.625, 0.40625, 0.40625],
        normalize_mode="percentile",
    ),
    inference=InferenceConfig(
        max_link_distance_um=7.0,
    ),
)

# Search for checkpoint
ckpt_candidates = [
    Path('src/best.pt'),
    Path('src/swa_final.pt'),
    Path('checkpoints_competition/best.pt'),
    Path('checkpoints/best.pt'),
]
ckpt_path = next((p for p in ckpt_candidates if p.exists()), None)

model = ResidualCellTrackingUNet.from_config(config)
if ckpt_path:
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt.get('model', ckpt))
    print(f'Loaded trained weights from: {ckpt_path}')
else:
    print('Warning: No checkpoint found, using default model initialization.')

model.to(device)
model.eval()
print('Model ready.')


In [ ]:
# ─── Inference Pipeline Helper ─────────────────────────────────────────────────────────────
def normalize_volume(volume, mode="percentile", percentile=99.5):
    vol = volume.astype(np.float32)
    if mode == "per_volume":
        vmax = float(vol.max())
        if vmax > 0:
            vol /= vmax
    elif mode == "percentile":
        vmax = float(np.percentile(vol, percentile))
        if vmax > 0:
            vol = np.clip(vol / vmax, 0.0, 1.0)
    return vol

def run_sample_inference(sample_path, model, config, timepoints_to_process=5):
    """Run cell detection and tracking over a subset of timepoints for visualization."""
    with open(os.path.join(sample_path, '0', 'zarr.json')) as f:
        meta = json.load(f)
        shape = meta['shape']
        chunk_shape = meta['chunk_grid']['chunk_shape'] if 'chunk_shape' in meta['chunk_grid'] else meta['chunk_grid']['configuration']['chunk_shape']
    
    num_timepoints = min(shape[0], timepoints_to_process)
    input_shape = tuple(shape[1:])
    min_shape = tuple(config.data.input_shape)
    
    def get_frame(t):
        chunk_path = os.path.join(sample_path, '0', 'c', str(t), '0', '0', '0')
        with open(chunk_path, 'rb') as f:
            decomp = blosc2.decompress(f.read())
        return np.frombuffer(decomp, dtype=np.uint16).reshape(chunk_shape).squeeze(0)
    
    raw_frames = []
    frame_detections = {}
    
    for t in range(num_timepoints):
        raw_vol = get_frame(t)
        raw_frames.append(raw_vol)
        
        frames = []
        for i in range(-3, 4):
            t_offset = max(0, min(t + i, shape[0] - 1))
            f = normalize_volume(get_frame(t_offset), mode=config.data.normalize_mode)
            frames.append(f)
        
        frame_window = np.stack(frames, axis=0)
        x = torch.from_numpy(frame_window).unsqueeze(0).unsqueeze(0).float().to(device)
        x_padded, (pz_b, py_b, px_b) = pad_spatial_tensor_if_needed(x, min_shape=min_shape)
        
        with torch.no_grad():
            outputs = model(x_padded, inference=True)
            ci     = outputs['cell_indicator'][0, 0].cpu().numpy()
            maxima = outputs['maxima'][0].cpu().numpy() if outputs['maxima'].ndim == 4 else outputs['maxima'][0, 0].cpu().numpy()
            mv     = outputs['movement_vectors'][0].cpu().numpy()
            
        actual_output_shape = ci.shape
        crop_offset_padded = compute_crop_offset(x_padded.shape[-3:], actual_output_shape)
        crop_offset = (
            crop_offset_padded[0] - pz_b,
            crop_offset_padded[1] - py_b,
            crop_offset_padded[2] - px_b,
        )
        
        nodes = extract_peaks_from_frame(ci, maxima, mv, t=t, threshold=config.model.nms_threshold, crop_offset=crop_offset)
        frame_detections[t] = nodes
        
    tracking_result = build_tracking_graph(
        frame_detections,
        max_distance_um=config.inference.max_link_distance_um,
        scale=tuple(config.data.scale),
        allow_divisions=True,
    )
    
    return raw_frames, frame_detections, tracking_result


In [ ]:
# ─── Visualization 1: Single Frame Cell Detections ──────────────────────────────────────────
def visualize_frame_predictions(raw_vol, nodes, timepoint=0, threshold=0.4):
    """Plot Maximum Intensity Projection (MIP) with detected cell centroids overlaid."""
    mip = np.max(raw_vol, axis=0)  # (Y, X) projection
    
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(mip, cmap="magma", origin="upper")
    
    xs, ys, confs = [], [], []
    for node in nodes:
        if node.confidence >= threshold:
            xs.append(node.x)
            ys.append(node.y)
            confs.append(node.confidence)
            
    sc = ax.scatter(xs, ys, c=confs, cmap="spring", s=45, edgecolors="white", linewidths=0.8, alpha=0.9)
    plt.colorbar(sc, ax=ax, label="Cell Indicator Confidence")
    
    ax.set_title(f"Cell Detections at Timepoint t={timepoint} ({len(xs)} cells detected)", fontsize=14, fontweight="bold")
    ax.set_xlabel("X (voxels)")
    ax.set_ylabel("Y (voxels)")
    plt.tight_layout()
    plt.show()


In [ ]:
# ─── Visualization 2: Multi-Frame Cell Tracking & Edges ─────────────────────────────────────
def visualize_tracking_trajectories(raw_frames, tracking_result, start_t=0, num_frames=3):
    """Plot consecutive frames with tracking links (edges) connecting parent-child cells across time."""
    fig, axes = plt.subplots(1, num_frames, figsize=(7 * num_frames, 7))
    if num_frames == 1:
        axes = [axes]
        
    node_dict = {node.node_id: node for node in tracking_result.nodes}
    t_edges = {t: [] for t in range(start_t, start_t + num_frames)}
    for src_id, tgt_id in tracking_result.edges:
        if src_id in node_dict and tgt_id in node_dict:
            src_node = node_dict[src_id]
            if start_t <= src_node.t < start_t + num_frames:
                t_edges[src_node.t].append((src_node, node_dict[tgt_id]))
                
    for idx, t in enumerate(range(start_t, start_t + num_frames)):
        if t >= len(raw_frames):
            break
        ax = axes[idx]
        mip = np.max(raw_frames[t], axis=0)
        ax.imshow(mip, cmap="gray", origin="upper")
        
        t_nodes = [n for n in tracking_result.nodes if n.t == t]
        ax.scatter([n.x for n in t_nodes], [n.y for n in t_nodes], color="cyan", s=35, zorder=3, label="Cells")
        
        edges = t_edges.get(t, [])
        for src, tgt in edges:
            ax.annotate(
                "",
                xy=(tgt.x, tgt.y),
                xytext=(src.x, src.y),
                arrowprops=dict(arrowstyle="->", color="lime", lw=1.8, alpha=0.95),
            )
            
        ax.set_title(f"Timepoint t={t} ({len(t_nodes)} cells, {len(edges)} links)", fontsize=13, fontweight="bold")
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        
    plt.tight_layout()
    plt.show()


In [ ]:
# ─── Run Inference & Display Overlay Predictions ────────────────────────────────────────────
search_dirs = ["train", "test", "../input/*", "."]
samples = []
for s_dir in search_dirs:
    samples.extend(glob.glob(f"{s_dir}/*.zarr"))
    samples.extend(glob.glob(f"{s_dir}/*/*.zarr"))

samples = sorted(list(set(samples)))

if samples:
    target_sample = samples[0]
    print(f"Processing sample volume: {target_sample}")
    raw_frames, frame_detections, tracking_result = run_sample_inference(
        target_sample, model, config, timepoints_to_process=5
    )
    
    print(f"Extracted {len(tracking_result.nodes)} detected cells and {len(tracking_result.edges)} tracking links.")
    
    # Render single frame detection overlay
    visualize_frame_predictions(raw_frames[0], frame_detections[0], timepoint=0)
    
    # Render multi-frame tracking trajectory edges
    visualize_tracking_trajectories(raw_frames, tracking_result, start_t=0, num_frames=3)
else:
    print("No .zarr dataset found in current directory. Place .zarr samples in train/ or test/ to run visualization.")
